# LangChain MCP Adapters - 实现 Anthropic MCP 与 LangChain/LangGraph 的兼容适配

## 一、关于 LangChain MCP Adapters
1、项目概览

轻量级适配器库，使 Anthropic Model Context Protocol (MCP) 工具能够兼容 LangChain 和 LangGraph 生态。
<img src="https://i-blog.csdnimg.cn/direct/0d3c88e3cc0b449b94b16f9618ebeed0.png">


2、相关链接资源
GitHub：https://github.com/langchain-ai/langchain-mcp-adapters

JS 版：https://github.com/langchain-ai/langchainjs-mcp-adapters

MCP协议文档：https://modelcontextprotocol.io/introduction

LangChain文档：https://python.langchain.com/docs/concepts/tools/

LangGraph文档：https://github.com/langchain-ai/langgraph

3、功能特性

MCP工具转换：将MCP工具转换为标准的LangChain工具，可直接用于LangGraph智能体

多服务器支持：提供客户端实现，支持同时连接多个MCP服务器并加载其工具


## 二、安装配置
`pip install langchain-mcp-adapters`


## 三、使用示例
1、基础用法

```shell
# 安装依赖
pip install langchain-mcp-adapters langgraph langchain-openai

# 设置环境变量
export OPENAI_API_KEY=<your_api_key>

```

### 服务端实现
创建一个能够进行数字加减乘除运算的MCP服务器。


In [ ]:
# math_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    """两数相加"""  
    return a + b 

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """两数相乘""" 
    return a * b 

if __name__ == "__main__":
    mcp.run(transport="stdio")


### 客户端调用


In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent

server_params = StdioServerParameters(
    command="python",
    args=["/path/to/math_server.py"],
)

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await load_mcp_tools(session)
        agent = create_react_agent("openai:gpt-4.1", tools)
        agent_response = await agent.ainvoke({"messages": "what's (3 + 5) x 12?"})


### 2、多服务器连接
服务端配置



In [ ]:
# weather_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Weather")

@mcp.tool()
async def get_weather(location: str) -> str:
    """获取指定地点天气"""
    return "It's always sunny in New York"

if __name__ == "__main__":
    mcp.run(transport="sse")


客户端调用

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

async with MultiServerMCPClient(
    {
        "math": {
            "command": "python",
            "args": ["/path/to/math_server.py"],
            "transport": "stdio",
        },
        "weather": {
            "url": "http://localhost:8000/sse",
            "transport": "sse",
        }
    }
) as client:
    agent = create_react_agent("openai:gpt-4.1", client.get_tools())
    math_response = await agent.ainvoke({"messages": "what's (3 + 5) x 12?"})
    weather_response = await agent.ainvoke({"messages": "what is the weather in nyc?"})


### 3、HTTP流式传输
MCP 现已支持可流式传输的 HTTP 传输协议。

启动 示例 服务器：

```shell
cd examples/servers/streamable-http-stateless/
uv run mcp-simple-streamablehttp-stateless --port 3000
```

要将其与 Python MCP SDK 的 streamablehttp_client 一起使用：

In [ ]:
# Use server from examples/servers/streamable-http-stateless/

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.tools import load_mcp_tools

async with streamablehttp_client("http://localhost:3000/mcp") as (read, write, _):
    async with ClientSession(read, write) as session:
        # Initialize the connection
        await session.initialize() 

        # Get tools
        tools = await load_mcp_tools(session)
        agent = create_react_agent("openai:gpt-4.1", tools)
        math_response = await agent.ainvoke({"messages": "what's (3 + 5) x 12?"})


与 MultiServerMCPClient 配合使用：

In [ ]:
# Use server from examples/servers/streamable-http-stateless/
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

async with MultiServerMCPClient(
    {
        "math": {
            "transport": "streamable_http",
            "url": "http://localhost:3000/mcp"
        },
    }
) as client:
    agent = create_react_agent("openai:gpt-4.1", client.get_tools())
    math_response = await agent.ainvoke({"messages": "what's (3 + 5) x 12?"})


### 4、LangGraph StateGraph集成


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition

from langchain.chat_models import init_chat_model
model = init_chat_model("openai:gpt-4.1")

async with MultiServerMCPClient(
    {
        "math": {
            "command": "python",
            # Make sure to update to the full absolute path to your math_server.py file
            "args": ["./examples/math_server.py"],
            "transport": "stdio",
        },
        "weather": {
            # make sure you start your weather server on port 8000
            "url": "http://localhost:8000/sse",
            "transport": "sse",
        }
    }
) as client:
    tools = client.get_tools()
    def call_model(state: MessagesState):
        response = model.bind_tools(tools).invoke(state["messages"])
        return {"messages": response}

    builder = StateGraph(MessagesState)
    builder.add_node(call_model)
    builder.add_node(ToolNode(tools))
    builder.add_edge(START, "call_model")
    builder.add_conditional_edges(
        "call_model",
        tools_condition,
    )
    builder.add_edge("tools", "call_model")
    graph = builder.compile()
    math_response = await graph.ainvoke({"messages": "what's (3 + 5) x 12?"})
    weather_response = await graph.ainvoke({"messages": "what is the weather in nyc?"})


配置langgraph.json，请确保将 make_graph 指定为图的入口点：

In [ ]:
{
  "dependencies": ["."],
  "graphs": {
    "agent": "./graph.py:make_graph"
  }
}
